# 04 - Interpretability

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
TARGET_COL = "DEATH_EVENT"

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

tables_dir = project_root / "results" / "tables"
figures_dir = project_root / "results" / "figures"

for folder in [tables_dir, figures_dir]:
    folder.mkdir(parents=True, exist_ok=True)


def load_table(file_name):
    return pd.read_csv(tables_dir / file_name)

In [ ]:
X_train_lr_with_time = load_table("X_train_lr_with_time.csv")
X_train_lr_without_time = load_table("X_train_lr_without_time.csv")

X_train_rf_with_time = load_table("X_train_rf_with_time.csv")
X_train_rf_without_time = load_table("X_train_rf_without_time.csv")

X_train_tree_with_time = load_table("X_train_tree_with_time.csv")
X_train_tree_without_time = load_table("X_train_tree_without_time.csv")

X_train_gb_with_time = load_table("X_train_gb_with_time.csv")
X_train_gb_without_time = load_table("X_train_gb_without_time.csv")

y_train = load_table("y_train.csv")[TARGET_COL]

In [ ]:
logistic_with_time = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
logistic_without_time = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)

random_forest_with_time = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE)
random_forest_without_time = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE)

decision_tree_with_time = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced", max_depth=4)
decision_tree_without_time = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced", max_depth=4)

gradient_boosting_with_time = GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100, learning_rate=0.05, max_depth=3)
gradient_boosting_without_time = GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100, learning_rate=0.05, max_depth=3)

logistic_with_time.fit(X_train_lr_with_time, y_train)
logistic_without_time.fit(X_train_lr_without_time, y_train)
random_forest_with_time.fit(X_train_rf_with_time, y_train)
random_forest_without_time.fit(X_train_rf_without_time, y_train)
decision_tree_with_time.fit(X_train_tree_with_time, y_train)
decision_tree_without_time.fit(X_train_tree_without_time, y_train)
gradient_boosting_with_time.fit(X_train_gb_with_time, y_train)
gradient_boosting_without_time.fit(X_train_gb_without_time, y_train)

In [ ]:
def logistic_coefficients(model, feature_names, time_version):
    table = pd.DataFrame({
        "feature": feature_names,
        "coefficient": model.coef_.ravel(),
    })
    table["absolute_coefficient"] = table["coefficient"].abs()
    table["time_version"] = time_version
    return table.sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)

coef_with_time = logistic_coefficients(logistic_with_time, X_train_lr_with_time.columns, "with_time")
coef_without_time = logistic_coefficients(logistic_without_time, X_train_lr_without_time.columns, "without_time")

coef_with_time.to_csv(tables_dir / "logistic_coefficients_with_time.csv", index=False)
coef_without_time.to_csv(tables_dir / "logistic_coefficients_without_time.csv", index=False)

coef_with_time.head(10)

In [ ]:
def plot_coefficients(table, file_name, title, top_n=12):
    plot_df = table.head(top_n).sort_values("coefficient")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_df["feature"], plot_df["coefficient"])
    ax.axvline(0, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Coefficient")
    plt.tight_layout()
    plt.savefig(figures_dir / file_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_coefficients(coef_with_time, "logistic_coefficients_with_time.png", "Logistic Regression Coefficients With Time")
plot_coefficients(coef_without_time, "logistic_coefficients_without_time.png", "Logistic Regression Coefficients Without Time")

In [ ]:
def feature_importance(model, feature_names, model_name, time_version):
    table = pd.DataFrame({
        "model": model_name,
        "feature": feature_names,
        "importance": model.feature_importances_,
        "time_version": time_version,
    })
    return table.sort_values("importance", ascending=False).reset_index(drop=True)

rf_importance_with_time = feature_importance(random_forest_with_time, X_train_rf_with_time.columns, "Random Forest", "with_time")
rf_importance_without_time = feature_importance(random_forest_without_time, X_train_rf_without_time.columns, "Random Forest", "without_time")
tree_importance_with_time = feature_importance(decision_tree_with_time, X_train_tree_with_time.columns, "Decision Tree", "with_time")
tree_importance_without_time = feature_importance(decision_tree_without_time, X_train_tree_without_time.columns, "Decision Tree", "without_time")
gb_importance_with_time = feature_importance(gradient_boosting_with_time, X_train_gb_with_time.columns, "Gradient Boosting", "with_time")
gb_importance_without_time = feature_importance(gradient_boosting_without_time, X_train_gb_without_time.columns, "Gradient Boosting", "without_time")

importance_tables = {
    "random_forest_importance_with_time.csv": rf_importance_with_time,
    "random_forest_importance_without_time.csv": rf_importance_without_time,
    "decision_tree_importance_with_time.csv": tree_importance_with_time,
    "decision_tree_importance_without_time.csv": tree_importance_without_time,
    "gradient_boosting_importance_with_time.csv": gb_importance_with_time,
    "gradient_boosting_importance_without_time.csv": gb_importance_without_time,
}

for file_name, table in importance_tables.items():
    table.to_csv(tables_dir / file_name, index=False)

rf_importance_with_time.head(10)

In [ ]:
def plot_importance(table, file_name, title, top_n=12):
    plot_df = table.head(top_n).sort_values("importance")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(plot_df["feature"], plot_df["importance"])
    ax.set_title(title)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(figures_dir / file_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_importance(rf_importance_with_time, "random_forest_importance_with_time.png", "Random Forest Importance With Time")
plot_importance(rf_importance_without_time, "random_forest_importance_without_time.png", "Random Forest Importance Without Time")
plot_importance(tree_importance_with_time, "decision_tree_importance_with_time.png", "Decision Tree Importance With Time")
plot_importance(tree_importance_without_time, "decision_tree_importance_without_time.png", "Decision Tree Importance Without Time")
plot_importance(gb_importance_with_time, "gradient_boosting_importance_with_time.png", "Gradient Boosting Importance With Time")
plot_importance(gb_importance_without_time, "gradient_boosting_importance_without_time.png", "Gradient Boosting Importance Without Time")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    decision_tree_with_time,
    feature_names=X_train_tree_with_time.columns,
    class_names=["No Death", "Death"],
    filled=True,
    rounded=True,
    max_depth=3,
    ax=ax,
)
plt.tight_layout()
plt.savefig(figures_dir / "decision_tree_plot_with_time.png", dpi=300, bbox_inches="tight")
plt.show()